# Pandas Reference Cheat Sheet
Quick-reference for pandas patterns used throughout the bootcamp.
All examples use the actual SAP datasets from `data/raw/`.


## 0. Setup

Load all five SAP datasets via the `analytics_bootcamp` helper and confirm shapes.


In [1]:
import numpy as np
import pandas as pd
from analytics_bootcamp.dataset import load_all

datasets = load_all()
materials   = datasets['materials']
sales       = datasets['sales_orders']
cost_center = datasets['cost_centers']
hr          = datasets['headcount']
bw          = datasets['bw_kpis']

for name, df in datasets.items():
    print(f"{name:14s} {df.shape}")


materials      (400, 10)
sales_orders   (500, 12)
cost_centers   (480, 10)
headcount      (350, 13)
bw_kpis        (480, 17)


## 1. Load / Save

Reading and writing dataframes. SAP extracts have leading zeros — always load as strings first.


In [2]:
from analytics_bootcamp.config import DATASETS

# Read everything as strings — preserves leading zeros (e.g. plant '0001')
raw = pd.read_csv(DATASETS['sales_orders'], dtype=str)

# Load only the columns you need — faster, less memory
slim = pd.read_csv(DATASETS['sales_orders'], usecols=['VBELN', 'MATNR', 'NETWR'])

# Parquet — faster + preserves dtypes vs CSV
# materials.to_parquet('materials.parquet', index=False)
# pd.read_parquet('materials.parquet')

# Standard CSV out
# sales.to_csv('out.csv', index=False)


## 2. Inspect

Quick health checks on a new dataframe.


In [3]:
sales.shape          # (rows, cols)
sales.head()
sales.dtypes
sales.describe()
sales.describe(include='object')


/tmp/ipykernel_8645/2143605714.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  sales.describe(include='object')


,VBELN,POSNR,KUNNR,MATNR,WAERK,VKORG,VTWEG,SPART,STATUS
count,500,500,500,500,500,500,500,500,500
unique,200,4,57,116,1,2,2,3,4
top,4500000,0010,CUST-0006,MAT-00077,USD,1000,20,00,DELIVERED
freq,4,200,27,9,500,265,263,178,284


In [4]:
# Memory footprint per column
sales.memory_usage(deep=True)


Index       132
VBELN     28000
POSNR     26500
KUNNR     29000
MATNR     29000
ERDAT      4000
NETWR      4000
MENGE      4000
WAERK     26000
VKORG     26500
VTWEG     25500
SPART     25500
STATUS    28328
dtype: int64

In [5]:
# Null counts per column
sales.isnull().sum()


VBELN     0
POSNR     0
KUNNR     0
MATNR     0
ERDAT     0
NETWR     0
MENGE     0
WAERK     0
VKORG     0
VTWEG     0
SPART     0
STATUS    0
dtype: int64

In [6]:
# Duplicate row count
sales.duplicated().sum()


np.int64(0)

In [7]:
# Cardinality of every column at once
sales.nunique()


VBELN     200
POSNR       4
KUNNR      57
MATNR     116
ERDAT     165
NETWR     500
MENGE     187
WAERK       1
VKORG       2
VTWEG       2
SPART       3
STATUS      4
dtype: int64

## 3. Basic Selection

Picking columns and rows by label, position, or pattern.


In [8]:
sales['NETWR']                 # Series
sales[['VBELN', 'MATNR', 'NETWR']]   # subset

# By position
sales.iloc[0]                  # first row
sales.iloc[0:5, 0:3]           # first 5 rows, first 3 cols

# By label
sales.loc[0, ['VBELN', 'NETWR']]


VBELN     4500000
NETWR    12405.56
Name: 0, dtype: object

In [9]:
# Regex column selection
sales.filter(regex='^V')       # columns starting with 'V'
materials.filter(regex='DATE$')


,LAST_MOVEMENT_DATE
0,2026-03-22
1,2024-05-07
2,2024-10-19
3,2025-06-29
4,2024-08-06
...,...
395,2024-05-08
396,2026-03-07
397,2025-05-05
398,2025-08-15


In [10]:
# Store the mask as a variable — re-usable and readable
mask = sales['NETWR'] > 10000
sales[mask].head()


,VBELN,POSNR,KUNNR,MATNR,ERDAT,NETWR,MENGE,WAERK,VKORG,VTWEG,SPART,STATUS
0,4500000,0010,CUST-0052,MAT-00056,2025-08-27,12405.56,102,USD,2000,10,00,DELIVERED
1,4500000,0020,CUST-0052,MAT-00079,2025-08-27,91853.66,51,USD,2000,10,00,DELIVERED
3,4500000,0040,CUST-0052,MAT-00028,2025-08-27,59200.34,53,USD,2000,10,00,DELIVERED
4,4500004,0010,CUST-0052,MAT-00015,2026-01-07,182542.75,68,USD,1000,20,10,OPEN
5,4500009,0010,CUST-0039,MAT-00046,2025-06-05,291620.94,118,USD,2000,20,20,OPEN


## 4. Boolean Filtering

The dominant filter style in the bootcamp.


In [11]:
sales[sales['NETWR'] > 0]
sales[(sales['NETWR'] > 0) & (sales['VKORG'] == '1000')]
sales[sales['STATUS'].isin(['OPEN', 'DELIVERED'])]

sales[sales['NETWR'].isna()]
sales[sales['NETWR'].notna()]

sales[sales['NETWR'].between(1000, 5000)]

# query() — handy interactively
sales.query("NETWR > 0 and VKORG == '1000'")


,VBELN,POSNR,KUNNR,MATNR,ERDAT,NETWR,MENGE,WAERK,VKORG,VTWEG,SPART,STATUS
4,4500004,0010,CUST-0052,MAT-00015,2026-01-07,182542.75,68,USD,1000,20,10,OPEN
15,4500026,0010,CUST-0018,MAT-00073,2026-03-19,590443.61,191,USD,1000,20,00,DELIVERED
16,4500026,0020,CUST-0018,MAT-00064,2026-03-19,288439.45,141,USD,1000,20,00,DELIVERED
17,4500029,0010,CUST-0035,MAT-00055,2025-04-11,79522.42,27,USD,1000,20,20,OPEN
20,4500035,0010,CUST-0011,MAT-00073,2025-02-07,191831.56,90,USD,1000,10,00,CANCELLED
...,...,...,...,...,...,...,...,...,...,...,...,...
492,4500590,0030,CUST-0047,MAT-00039,2024-10-23,340143.04,114,USD,1000,10,10,CANCELLED
493,4500591,0010,CUST-0007,MAT-00079,2025-10-11,169236.35,158,USD,1000,10,00,DELIVERED
494,4500591,0020,CUST-0007,MAT-00073,2025-10-11,145616.06,157,USD,1000,10,00,DELIVERED
495,4500591,0030,CUST-0007,MAT-00030,2025-10-11,165064.48,60,USD,1000,10,00,DELIVERED


## 5. String Operations

SAP field values are often messy — strip, pad, normalize.


In [12]:
# Strip whitespace (common in SAP extracts)
materials['MATNR'] = materials['MATNR'].str.strip()

# Pad material numbers (SAP convention is 18 chars)
materials['MATNR_PADDED'] = materials['MATNR'].str.zfill(18)

# Upper / lower
materials['MAKTX_UPPER'] = materials['MAKTX'].str.upper()


In [13]:
# Contains / startswith / endswith
materials[materials['MATNR'].str.startswith('MAT')]
materials[materials['MAKTX'].str.contains('Gear', case=False, na=False)]


,MATNR,MAKTX,WERKS,LGORT,LABST,EINHEIT,STPRS,LAST_MOVEMENT_DATE,MATERIAL_TYPE,MRPTYPE,MATNR_PADDED,MAKTX_UPPER
0,MAT-00001,Sub-Assembly Gear Box,2000,0001,303,EA,63.34,2026-03-22,HALB,PD,000000000MAT-00001,SUB-ASSEMBLY GEAR BOX
14,MAT-00015,Sub-Assembly Gear Box,1000,WH02,249,PC,28.92,2025-12-07,HALB,VB,000000000MAT-00015,SUB-ASSEMBLY GEAR BOX
20,MAT-00021,Sub-Assembly Gear Box,3000,0002,483,PC,43.44,2024-09-06,HALB,PD,000000000MAT-00021,SUB-ASSEMBLY GEAR BOX
81,MAT-00082,Sub-Assembly Gear Box,3000,WH01,168,EA,26.10,2025-10-24,HALB,PD,000000000MAT-00082,SUB-ASSEMBLY GEAR BOX
92,MAT-00093,Sub-Assembly Gear Box,1000,0001,228,PC,226.99,2025-05-27,HALB,VB,000000000MAT-00093,SUB-ASSEMBLY GEAR BOX
120,MAT-00021,Sub-Assembly Gear Box,2000,WH01,463,PC,43.44,2024-11-12,HALB,PD,000000000MAT-00021,SUB-ASSEMBLY GEAR BOX
159,MAT-00021,Sub-Assembly Gear Box,1000,0002,434,PC,43.44,2024-08-07,HALB,PD,000000000MAT-00021,SUB-ASSEMBLY GEAR BOX
189,MAT-00093,Sub-Assembly Gear Box,3000,WH02,21,PC,226.99,2026-03-23,HALB,VB,000000000MAT-00093,SUB-ASSEMBLY GEAR BOX
223,MAT-00015,Sub-Assembly Gear Box,3000,WH02,157,PC,28.92,2024-05-17,HALB,VB,000000000MAT-00015,SUB-ASSEMBLY GEAR BOX
228,MAT-00015,Sub-Assembly Gear Box,2000,0001,225,PC,28.92,2025-06-04,HALB,VB,000000000MAT-00015,SUB-ASSEMBLY GEAR BOX


In [14]:
# Extract with regex — grab the numeric suffix from MAT-00056
materials['MAT_NUM'] = materials['MATNR'].str.extract(r'-(\d+)$')

# Split and expand
parts = materials['MATNR'].str.split('-', expand=True)
parts.columns = ['prefix', 'suffix']
parts.head()


,prefix,suffix
0,MAT,00001
1,MAT,00002
2,MAT,00003
3,MAT,00004
4,MAT,00005


In [15]:
# Replace with regex — collapse repeated whitespace
materials['MAKTX_CLEAN'] = materials['MAKTX'].str.replace(r'\s+', ' ', regex=True)


## 6. Type Casting

Critical for SAP data — raw extracts are all strings until you coerce them.


In [16]:
# Reload as strings to demonstrate
raw = pd.read_csv(DATASETS['sales_orders'], dtype=str)

raw['NETWR'] = pd.to_numeric(raw['NETWR'], errors='coerce')
raw['MENGE'] = pd.to_numeric(raw['MENGE'], errors='coerce')
raw['ERDAT'] = pd.to_datetime(raw['ERDAT'], errors='coerce')


In [17]:
# Cast multiple columns in one shot
raw_cc = pd.read_csv(DATASETS['cost_centers'], dtype=str)
numeric_cols = ['ACTUAL_AMT', 'PLAN_AMT', 'GJAHR', 'PERIOD']
raw_cc[numeric_cols] = raw_cc[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Check what survived the cast — non-zero means coerce failures
raw_cc[numeric_cols].isnull().sum()


ACTUAL_AMT    0
PLAN_AMT      0
GJAHR         0
PERIOD        0
dtype: int64

## 7. Create / Modify Columns

Derive new columns with vectorized ops, binning, mapping, or apply (sparingly).


In [18]:
# Simple derived column
materials['INV_VALUE'] = materials['LABST'] * materials['STPRS']

# Fill / replace
materials['STPRS'] = materials['STPRS'].fillna(0)

# Drop / rename
slim = materials.drop(columns=['LGORT']).rename(columns={'MAKTX': 'description'})
slim.head(2)


,MATNR,description,WERKS,LABST,EINHEIT,STPRS,LAST_MOVEMENT_DATE,MATERIAL_TYPE,MRPTYPE,MATNR_PADDED,MAKTX_UPPER,MAT_NUM,MAKTX_CLEAN,INV_VALUE
0,MAT-00001,Sub-Assembly Gear Box,2000,303,EA,63.34,2026-03-22,HALB,PD,000000000MAT-00001,SUB-ASSEMBLY GEAR BOX,00001,Sub-Assembly Gear Box,19192.02
1,MAT-00002,Epoxy Resin Type-A,1000,579,KG,44.67,2024-05-07,ROH,PD,000000000MAT-00002,EPOXY RESIN TYPE-A,00002,Epoxy Resin Type-A,25863.93


In [19]:
# np.where — vectorized if/else (much faster than apply)
days_since = (pd.Timestamp.today() - materials['LAST_MOVEMENT_DATE']).dt.days
materials['aging_bucket'] = np.where(days_since <= 30, '0-30',
                            np.where(days_since <= 60, '31-60',
                            np.where(days_since <= 90, '61-90', '90+')))
materials['aging_bucket'].value_counts()


aging_bucket
90+      372
31-60     12
61-90     12
0-30       4
Name: count, dtype: int64

In [20]:
# pd.cut — bin a continuous variable into labelled buckets
materials['days_since'] = days_since
materials['bucket'] = pd.cut(materials['days_since'],
                             bins=[0, 30, 60, 90, float('inf')],
                             labels=['0-30', '31-60', '61-90', '90+'])


In [21]:
# map — lookup from a dict
material_type_names = {'ROH': 'Raw', 'HALB': 'Semi-Finished', 'FERT': 'Finished'}
materials['mat_type_name'] = materials['MATERIAL_TYPE'].map(material_type_names)


In [22]:
# apply — row-level function (slow; reserve for cases you can't vectorize)
materials['size_flag'] = materials['STPRS'].apply(lambda x: 'expensive' if x > 100 else 'cheap')


## 8. Groupby / Aggregation

Named aggregation is the bootcamp standard — readable columns out the back.


In [23]:
# Multi-key groupby — common SAP pattern
cost_center.groupby(['KOSTL', 'GJAHR', 'PERIOD'])['ACTUAL_AMT'].sum().head()


KOSTL    GJAHR  PERIOD
CC-1001  2024   2         60771.37
CC-1002  2024   12        79527.47
CC-1003  2023   2         59298.38
                5         66213.62
         2024   3         89967.75
Name: ACTUAL_AMT, dtype: float64

In [24]:
# Named agg over multiple columns
cost_center.groupby('KOSTL').agg(
    actual=('ACTUAL_AMT', 'sum'),
    plan=('PLAN_AMT', 'sum'),
    periods=('PERIOD', 'nunique'),
).head()


,actual,plan,periods
KOSTL,,,
CC-1001,60771.37,74626.23,1
CC-1002,79527.47,63492.04,1
CC-1003,346175.30,338168.66,5
CC-1004,132319.78,140830.65,2
CC-1005,74589.01,72349.43,1


In [25]:
# Chain assign() onto the aggregate for computed columns
summary = (
    cost_center
    .groupby('KOSTL')
    .agg(actual=('ACTUAL_AMT', 'sum'), plan=('PLAN_AMT', 'sum'))
    .assign(
        variance=lambda d: d['actual'] - d['plan'],
        variance_pct=lambda d: (d['actual'] - d['plan']) / d['plan'] * 100,
    )
)
summary.head()


,actual,plan,variance,variance_pct
KOSTL,,,,
CC-1001,60771.37,74626.23,-13854.86,-18.565671
CC-1002,79527.47,63492.04,16035.43,25.255812
CC-1003,346175.30,338168.66,8006.64,2.367647
CC-1004,132319.78,140830.65,-8510.87,-6.043336
CC-1005,74589.01,72349.43,2239.58,3.095505


## 9. Window Functions

Ranks, running totals, lags, rolling averages — pandas equivalents of SQL window functions.


In [26]:
# Rank within group
cc = cost_center.copy()
cc['rank'] = cc.groupby('KOSTL')['ACTUAL_AMT'].rank(method='dense', ascending=False)

# Running total (cumsum within group)
cc = cc.sort_values(['KOSTL', 'GJAHR', 'PERIOD'])
cc['running_total'] = cc.groupby('KOSTL')['ACTUAL_AMT'].cumsum()
cc[['KOSTL', 'GJAHR', 'PERIOD', 'ACTUAL_AMT', 'running_total']].head()


,KOSTL,GJAHR,PERIOD,ACTUAL_AMT,running_total
95,CC-1001,2024,2,40456.06,40456.06
96,CC-1001,2024,2,3612.42,44068.48
97,CC-1001,2024,2,3156.92,47225.40
98,CC-1001,2024,2,10263.01,57488.41
99,CC-1001,2024,2,3282.96,60771.37


In [27]:
# LAG — previous period value
cc['prev_period'] = cc.groupby('KOSTL')['ACTUAL_AMT'].shift(1)
cc['period_change'] = cc['ACTUAL_AMT'] - cc['prev_period']


In [28]:
# Rolling 3-period average within group
cc['rolling_3'] = cc.groupby('KOSTL')['ACTUAL_AMT'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)


In [29]:
# Percent of group total
cc['pct_of_total'] = cc['ACTUAL_AMT'] / cc.groupby('KOSTL')['ACTUAL_AMT'].transform('sum')
cc[['KOSTL', 'ACTUAL_AMT', 'pct_of_total']].head()


,KOSTL,ACTUAL_AMT,pct_of_total
95,CC-1001,40456.06,0.665709
96,CC-1001,3612.42,0.059443
97,CC-1001,3156.92,0.051947
98,CC-1001,10263.01,0.168879
99,CC-1001,3282.96,0.054021


## 10. Joins / Merging

The SAP cross-dataset keys in this repo:
- `MATNR` — links `materials` ↔ `sales_orders` ↔ `bw_kpis`
- `KOSTL` — links `cost_centers` ↔ `headcount`
- `WERKS` — plant, appears in `materials` and `headcount`


In [30]:
# Standard left join — sales enriched with material master
merged = pd.merge(
    sales,
    materials[['MATNR', 'WERKS', 'LABST']],
    on='MATNR', how='left',
)
merged.head(2)


,VBELN,POSNR,KUNNR,MATNR,ERDAT,NETWR,MENGE,WAERK,VKORG,VTWEG,SPART,STATUS,WERKS,LABST
0,4500000,0010,CUST-0052,MAT-00056,2025-08-27,12405.56,102,USD,2000,10,00,DELIVERED,1000,629
1,4500000,0010,CUST-0052,MAT-00056,2025-08-27,12405.56,102,USD,2000,10,00,DELIVERED,3000,458


In [31]:
# Join on multiple keys
hr_by_cc = pd.merge(
    cost_center, hr[['KOSTL', 'PERNR', 'SALARY']],
    on='KOSTL', how='inner',
)

# Merge with different column names on each side
# pd.merge(df1, df2, left_on='material_number', right_on='MATNR', how='left')


In [32]:
# Check merge quality — how many matched?
print(f"Left rows:    {len(sales)}")
print(f"Merged rows:  {len(merged)}")
print(f"Unmatched:    {merged['WERKS'].isna().sum()}")

# indicator=True gives you a _merge column showing left_only / both / right_only
audit = pd.merge(sales, materials[['MATNR']], on='MATNR', how='left', indicator=True)
audit['_merge'].value_counts()


Left rows:    500
Merged rows:  1641
Unmatched:    0


_merge
both          1641
left_only        0
right_only       0
Name: count, dtype: int64

## 11. Duplicate Handling

Find, view, and drop duplicate rows.


In [33]:
# Count dupes
materials.duplicated().sum()
materials.duplicated(subset=['MATNR', 'WERKS']).sum()


np.int64(116)

In [34]:
# View all dupe rows (keep=False shows every copy)
materials[materials.duplicated(subset=['MATNR'], keep=False)].head()


,MATNR,MAKTX,WERKS,LGORT,LABST,EINHEIT,STPRS,LAST_MOVEMENT_DATE,MATERIAL_TYPE,MRPTYPE,MATNR_PADDED,MAKTX_UPPER,MAT_NUM,MAKTX_CLEAN,INV_VALUE,aging_bucket,days_since,bucket,mat_type_name,size_flag
0,MAT-00001,Sub-Assembly Gear Box,2000,0001,303,EA,63.34,2026-03-22,HALB,PD,000000000MAT-00001,SUB-ASSEMBLY GEAR BOX,00001,Sub-Assembly Gear Box,19192.02,31-60,53,31-60,Semi-Finished,cheap
1,MAT-00002,Epoxy Resin Type-A,1000,0002,579,KG,44.67,2024-05-07,ROH,PD,000000000MAT-00002,EPOXY RESIN TYPE-A,00002,Epoxy Resin Type-A,25863.93,90+,737,90+,Raw,cheap
2,MAT-00003,Plastic Granules HDPE,3000,WH01,98,KG,9.51,2024-10-19,ROH,PD,000000000MAT-00003,PLASTIC GRANULES HDPE,00003,Plastic Granules HDPE,931.98,90+,572,90+,Raw,cheap
3,MAT-00004,Control Panel CP-100,1000,0002,36,EA,4265.44,2025-06-29,FERT,ND,000000000MAT-00004,CONTROL PANEL CP-100,00004,Control Panel CP-100,153555.84,90+,319,90+,Finished,expensive
4,MAT-00005,Control Panel CP-100,3000,0001,137,PC,1875.20,2024-08-06,FERT,VB,000000000MAT-00005,CONTROL PANEL CP-100,00005,Control Panel CP-100,256902.40,90+,646,90+,Finished,expensive


In [35]:
# Drop dupes — keep first occurrence
dedup = materials.drop_duplicates(subset=['MATNR', 'WERKS'], keep='first')

# Keep the row with the latest movement date per material
latest = (
    materials
    .sort_values('LAST_MOVEMENT_DATE')
    .drop_duplicates(subset=['MATNR'], keep='last')
)


## 12. Reshape

Wide ↔ long transformations.


In [36]:
# long -> wide
pivot = materials.pivot_table(
    index='MATNR', columns='WERKS', values='LABST',
    aggfunc='sum', fill_value=0,
)
pivot.head()


WERKS,1000,2000,3000
MATNR,,,
MAT-00001,334,303,0
MAT-00002,579,233,354
MAT-00003,0,481,98
MAT-00004,36,0,722
MAT-00005,675,114,137


In [37]:
# wide -> long
long = pivot.reset_index().melt(
    id_vars=['MATNR'], var_name='WERKS', value_name='LABST',
)
long.head()


,MATNR,WERKS,LABST
0,MAT-00001,1000,334
1,MAT-00002,1000,579
2,MAT-00003,1000,0
3,MAT-00004,1000,36
4,MAT-00005,1000,675


In [38]:
# Crosstab — quick frequency table
pd.crosstab(materials['WERKS'], materials['MATERIAL_TYPE'])


MATERIAL_TYPE,FERT,HALB,ROH
WERKS,,,
1000,44,47,51
2000,38,39,52
3000,37,42,50


In [39]:
# unstack — pivot a MultiIndex level into columns
period_by_cc = (
    cost_center
    .groupby(['KOSTL', 'PERIOD'])['ACTUAL_AMT'].sum()
    .unstack('PERIOD')
)
period_by_cc.head()


PERIOD,1,2,3,4,5,6,7,8,9,10,11,12
KOSTL,,,,,,,,,,,,
CC-1001,NaN,60771.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CC-1002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,79527.47
CC-1003,NaN,59298.38,89967.75,75381.13,66213.62,NaN,NaN,NaN,55314.42,NaN,NaN,NaN
CC-1004,NaN,NaN,NaN,NaN,NaN,NaN,63467.94,NaN,68851.84,NaN,NaN,NaN
CC-1005,74589.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 13. Datetime

Parsing SAP date formats and doing period arithmetic.


In [40]:
# Two common SAP date formats
# pd.to_datetime(s, format='%Y%m%d')    # YYYYMMDD compact
# pd.to_datetime(s, format='%d.%m.%Y')  # DD.MM.YYYY European

sales['year']    = sales['ERDAT'].dt.year
sales['month']   = sales['ERDAT'].dt.month
sales['quarter'] = sales['ERDAT'].dt.quarter
sales['weekday'] = sales['ERDAT'].dt.day_name()


In [41]:
# Period arithmetic
sales['days_since'] = (pd.Timestamp.today() - sales['ERDAT']).dt.days
sales['months_since'] = (
    (pd.Timestamp.today().year - sales['ERDAT'].dt.year) * 12
    + (pd.Timestamp.today().month - sales['ERDAT'].dt.month)
)


In [42]:
# Filter by date range
sales[sales['ERDAT'].between('2025-01-01', '2025-12-31')].shape


(324, 18)

## 14. Method Chaining

Clean pipeline style — every step is a method on the previous result.


In [43]:
result = (
    cost_center
    .assign(
        variance     = lambda d: d['ACTUAL_AMT'] - d['PLAN_AMT'],
        variance_pct = lambda d: (d['ACTUAL_AMT'] - d['PLAN_AMT']) / d['PLAN_AMT'] * 100,
    )
    .query("GJAHR == 2023")
    .groupby('KOSTL')
    .agg(
        actual=('ACTUAL_AMT', 'sum'),
        plan=('PLAN_AMT', 'sum'),
        variance=('variance', 'sum'),
    )
    .sort_values('variance')
    .head(10)
)
result


,actual,plan,variance
KOSTL,,,
CC-1009,120230.91,148307.24,-28076.33
CC-1046,181630.66,208208.18,-26577.52
CC-1021,111158.38,135565.76,-24407.38
CC-1023,55146.10,72785.65,-17639.55
CC-1006,66501.89,72216.72,-5714.83
CC-1003,125512.00,129380.82,-3868.82
CC-1017,57991.82,61640.19,-3648.37
CC-1033,142407.61,144969.05,-2561.44
CC-1043,75686.75,74871.87,814.88


## 15. DuckDB ↔ pandas

Use DuckDB for SQL-shaped work, pandas for cleaning and reshaping. The repo's `duckdb_utils` registers the raw CSVs as views.


In [44]:
from analytics_bootcamp.duckdb_utils import get_connection, query_df

con = get_connection()

# SQL -> DataFrame
df = query_df("""
    SELECT KOSTL, SUM(ACTUAL_AMT) AS total
    FROM cost_centers
    GROUP BY KOSTL
    ORDER BY total DESC
    LIMIT 10
""", con)
df


,KOSTL,total
0,CC-1038,381754.61
1,CC-1003,346175.30
2,CC-1011,335520.37
3,CC-1028,324753.18
4,CC-1010,300690.89
5,CC-1044,300143.22
6,CC-1036,293300.32
7,CC-1035,278571.90
8,CC-1016,274864.03
9,CC-1046,264405.99


In [45]:
# DataFrame -> DuckDB (register a pandas df as a view)
con.register('my_df', df)
query_df("SELECT * FROM my_df WHERE total > 1000000", con)


,KOSTL,total


**When to use which:**
- **DuckDB** — aggregations, joins, window functions, anything SQL-shaped
- **pandas** — string cleaning, reshaping, `apply()`, visualization prep


## 16. Performance Tips

Vectorized beats apply; apply beats iterrows. Never use `iterrows` on large frames.


In [46]:
# SLOW — apply for simple arithmetic
# sales['x2'] = sales['NETWR'].apply(lambda x: x * 2)

# FAST — vectorized
sales['x2'] = sales['NETWR'] * 2


In [47]:
# np.where beats apply for conditional logic
sales['big'] = np.where(sales['NETWR'] > 10000, 'large', 'small')


In [48]:
# query() — readable filters on large frames
cost_center.query("KOSTL == 'CC-1016' and GJAHR == 2023").head()


,BUKRS,KOSTL,KTEXT,GJAHR,PERIOD,KSTAR,KSTAR_DESC,ACTUAL_AMT,PLAN_AMT,CURRENCY
0,1000,CC-1016,Operations Administration,2023,1,400000,Salaries,53674.80,46877.65,USD
1,1000,CC-1016,Operations Administration,2023,1,410000,Travel,5738.65,5663.07,USD
2,1000,CC-1016,Operations Administration,2023,1,420000,Supplies,2683.29,2669.67,USD
3,1000,CC-1016,Operations Administration,2023,1,430000,Depreciation,10490.94,6923.65,USD
4,1000,CC-1016,Operations Administration,2023,1,440000,Utilities,3465.04,2867.15,USD


In [49]:
# Categorical dtype for low-cardinality strings — big memory savings
before = materials.memory_usage(deep=True).sum()
materials['WERKS'] = materials['WERKS'].astype('category')
after = materials.memory_usage(deep=True).sum()
print(f"before: {before:,}  after: {after:,}  saved: {before - after:,} bytes")


before: 340,204  after: 319,563  saved: 20,641 bytes
